This notebooks aim is to answer the question, which terms actually exist in the picrust2 results of the data.

In [2]:
import os
import sys
from pathlib import Path
import pickle
import pyarrow

# Data processing and analysis
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath('..'))  # Ensures the project root is in Python's search path

if Path("/kaggle").exists():
    
    # Create directory structure
    !mkdir -p corrosion_scoring
    
    # Download the necessary files each session always
    !wget -O corrosion_scoring/__init__.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/__init__.py
    !wget -O corrosion_scoring/global_terms.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/global_terms.py
    !wget -O corrosion_scoring/scoring_system.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/scoring_system.py
    !wget -O corrosion_scoring/term_processor.py https://raw.githubusercontent.com/MagicAlex238/2_Micro/main/corrosion_scoring_root/corrosion_scoring/term_processor.py
    # Add current directory to path
    import sys
    sys.path.append(os.getcwd())
    
    # Import package
    import corrosion_scoring as cs
else:
    print("Running in local (VSCode) environment")
    
    ## When in vscode local env first time only
    #!pip install git+https://github.com/MagicAlex238/2_Micro.git#subdirectory=corrosion_scoring_root
    import corrosion_scoring as cs

Running in local (VSCode) environment


In [3]:
# environment check
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    # For Kaggle # Whole filtered Data
    eccontri_path = Path("/kaggle/input/eccontri-uniprot-enriched/ECcontri_Uniprot_enriched.parquet")
    ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)
    # Directory to output large files 
    large_dir =  Path("/kaggle/working/")
    # Directory to output large files # eccontris, compilated db

else:              
    # For Vscode 
    # large galaxies input and output #large size dir for large files hosted instead in kaggle
    large_dir = Path("/home/beatriz/MIC")
    # Directory to output large files
    output_large = large_dir / "output_large"
    # Whole filtered Data
    eccontri_path = output_large / 'ECcontri_Uniprot_enriched.parquet'


In [4]:
# Whole filtered Data
ECcontri_Uniprot_enriched = pd.read_parquet(eccontri_path)

In [ ]:
def validate_terms(df, global_terms_list):
    """
    Checks which terms from a list of global dictionaries exist in the data.
    
    Args: df : dataframe with data to check
          global_terms: global list of dictionaries with metal_terms, pathways, mechanisms, functional categories, organic proceses and keywords
    
    Returns:  dict: {category: [found_terms]}
    """
    df = df.copy()
    found = {}
    cols_terms = [
            'idx', 'Genus', 'protein_name', 'EC', 'enzyme_names',
            'enzyme_class', 'pathways', 'hierarchy', 'metals_consolidated',
            'corrosion_mechanisms', 'corrosion_relevance', 'functional_categories', 
            'corrosion_keyword_groups', 'corrosion_synergies', 'organic_processes'
        ]

    for d in global_terms_list:
        for category, terms in d.items():
            if isinstance(terms, dict):
                # Handle functional_categories special case, nested with scores
                if 'terms' in terms and 'score' in terms:
                    # This is functional_categories format: {'terms': [...], 'score': 1.5}
                    existing = []
                    for term in terms['terms']:  # Access the 'terms' key
                        for col in cols_terms:
                            if col in df.columns:
                                if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                    existing.append(term)
                                    break
                    if existing:
                        found[category] = existing
                else:
                    # Handle other nested dictionaries
                    for subcategory, subterms in terms.items():
                        if isinstance(subterms, list):  # Making sure it's a list
                            existing = []
                            for term in subterms:
                                for col in cols_terms:
                                    if col in df.columns:
                                        if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                            existing.append(term)
                                            break
                            if existing:
                                found[f"{category}.{subcategory}"] = existing
            else:
                # Handle simple lists
                existing = []
                for term in terms:
                    for col in cols_terms:
                        if col in df.columns:
                            if df[col].fillna('').astype(str).str.contains(term, case=False, regex=False).any():
                                existing.append(term)
                                break
                if existing:
                    found[category] = existing
    
    return found
#sample= ECcontri_Uniprot_enriched.sample(n=15000)

In [ ]:
real_terms = validate_terms(ECcontri_Uniprot_enriched, [
    cs.metal_terms,
    cs.corrosion_mechanisms,
    cs.pathway_categories,
    cs.organic_categories,
    cs.corrosion_synergies,
    cs.functional_categories,
    cs.corrosion_keyword_groups
])

In [6]:
# Saving the new dataframe
is_kaggle = os.path.exists("/kaggle")
if is_kaggle:
    rt_path = large_dir / 'real_terms.pkl'
else:
    rt_path = output_large / 'real_terms.pkl'

In [ ]:
with open(rt_path, 'wb') as f:
    pickle.dump(real_terms, f)

In [10]:
# Reading the dictionaries
with open(rt_path, 'rb') as f:
    real_terms= pickle.load(f)
# Print in compact format
for category, terms in real_terms.items():
    terms_str = ', '.join(terms)
    print(f"'{category}': [{terms_str}]")

'iron': [Fe3+, iron, ferric, heme, iron-sulfur, siderophore, ferritin, ferredoxin, rubredoxin, iron-sulfur cluster]
'manganese': [manganese, mn]
'copper': [Cu+, copper, cupric]
'nickel': [Ni2+, nickel]
'cobalt': [cobalt, cobalamin, vitamin B12]
'magnesium': [magnesium]
'calcium': [Ca2+, calcium]
'Mo': [Mo, molybdenum, molybdopterin, molybdenum cofactor, molybdate]
'V5+': [V5+, vanadium]
'Al3+': [Al3+]
'Cr3+': [Cr3+, chromate]
'zinc': [Zn2+, zinc]
'sodium': [sodium]
'potassium': [potassium]
'selenium': [selenium, Se, selenocysteine, selenoprotein, selenite, selenate]
'lead': [lead]
'arsenic': [arsenic, arsenite, arsenate]
'mercury': [mercury, mercuric]
'phosphate': [phosphate, orthophosphate]
'nitrate': [NO3-, nitrate]
'nitrite': [nitrite]
'chloride': [Cl-, chloride]
'sulfate': [sulfate]
'sulfide': [sulfide, desulfovibrio, h2s]
'thiosulfate': [thiosulfate]
'oxygen': [O2, oxygen, oxidase, superoxide, peroxide]
'hydrogen': [hydrogenase, h2]
'organics': [methane, methane, methanogenesis, f

The process undergone for the scoring system has been iterative and during the first iteration it was noticed that pathways and mechanisms are highly interconnected due to the fact that one bacterium expresses multiple proteins across many pathways and mechanisms. Pathway and mechanism categories exhibit substantial overlap due to multi-protein expression patterns within individual bacterial strains. As a response to this iteration, functional_categories were designed to reconcile this complexity by grouping related processes and make it more about functional metabolism. On a following iteration more modern terms were introduced and diverse terms were tried. Ultimately, it was evident that it was necesary a reality check to validate the terms with the bioinformatics annotations. 
An script was done to critically evaluate the real terms possible to be mined from the compiled database after the enrichment of the data (ECcontri_Uniprot_enriched) with ec_records. The script compared the enriched data with the global terms which teoretically proposed the dictionaries grouped by categories namely: metal_terms, mechanisms, pathways, functional_categories, organic_processes, synergies and keywords.
Analysis of the enriched data's real terms, prompt to redefine the corrosion scoring system by eliminating non-existent terms and consolidating overlapping categorical structures. Yet some theoretical terms are left for teoretical completness. A manual curation was done to reasign categories for efficient computational resource allocation.
The categories to consolidate are: corrosion_synergies,metal_terms, functional_categories, mechanisms and pathways. The categories to remove are: corrosion_keyword_groups and organic_processes. A hierarchy of the categories is stablished, this allows a first term algorithm to prioritize the terms allocated, in order to prevent duplicates.
The scoring takes into account only corrosion_synergies,metal_terms and functional_categories.
